In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

In [2]:
REGION_TO_ELECTRODES = {
    'Middle_Finger': ['E1', 'E2', 'E3'],
    'Hand': ['E4', 'E5', 'E6', 'E7'],
    'Forearm': ['E8', 'E9', 'E10', 'E11', 'E12', 'E13'],
    'Arm': ['E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20']
}

ELECTRODE_TO_REGION = {
    electrode: region
    for region, electrodes in REGION_TO_ELECTRODES.items()
    for electrode in electrodes
}

REGION_TO_LABEL = {
    'Middle_Finger': 0,
    'Hand': 1,
    'Forearm': 2,
    'Arm': 3
}

LABEL_TO_REGION = {v: k for k, v in REGION_TO_LABEL.items()}

print("Region-to-Electrode Mapping:")
for region, electrodes in REGION_TO_ELECTRODES.items():
    print(f"  {region} (Class {REGION_TO_LABEL[region]}): {electrodes} ({len(electrodes)} electrodes)")

Region-to-Electrode Mapping:
  Middle_Finger (Class 0): ['E1', 'E2', 'E3'] (3 electrodes)
  Hand (Class 1): ['E4', 'E5', 'E6', 'E7'] (4 electrodes)
  Forearm (Class 2): ['E8', 'E9', 'E10', 'E11', 'E12', 'E13'] (6 electrodes)
  Arm (Class 3): ['E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20'] (7 electrodes)


In [12]:
BIDS_ROOT = Path(r"../../data/BIDS-somatosensory/BIDS-somatosensory")
DERIVATIVES = BIDS_ROOT / "derivatives" / "fmriprep"

subjects = sorted([d.name for d in DERIVATIVES.iterdir() if d.is_dir() and d.name.startswith("sub-")])
session = "ses-01"
task = "task-S1Map"
space = "MNI152NLin2009cAsym"
n_runs_per_subject = 4

HRF_DELAY = 6.0
WINDOW = 1

subject_ref = subjects[0]
bold_json = DERIVATIVES / subject_ref / session / "func" / (
    f"{subject_ref}_{session}_{task}_run-1_space-{space}_desc-preproc_bold.json"
)

with open(bold_json, "r", encoding="utf-8") as f:
    tr = float(json.load(f)["RepetitionTime"])

print(f"Subject: {subject_ref}")
print(f"Runs: {n_runs_per_subject}")
print(f"TR: {tr} s")

Subject: sub-p0002
Runs: 4
TR: 2.0 s


In [13]:
RESULTS_BASE_DIR = Path("results/4_Classes_ANN_MultiSubject")
RESULTS_BASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base results folder: {RESULTS_BASE_DIR.resolve()}")
print(f"Subjects to process ({len(subjects)}): {subjects}")

Base results folder: /home/duarte/Desktop/Tese/Mapping_Tese/mapping_tese/notebooks/ANN/results/4_Classes_ANN_MultiSubject
Subjects to process (9): ['sub-p0002', 'sub-p0004', 'sub-p0005', 'sub-p0008', 'sub-p0009', 'sub-p0010', 'sub-p0016', 'sub-p0023', 'sub-p0024']


In [14]:
from nilearn.datasets import fetch_atlas_destrieux_2009
from nilearn.image import load_img, index_img, resample_to_img, new_img_like
from nilearn.maskers import NiftiMasker

atlas = fetch_atlas_destrieux_2009()

first_run_path = (
    DERIVATIVES / subject_ref / session / "func" /
    f"{subject_ref}_{session}_{task}_run-1_space-{space}_desc-preproc_bold.nii"
)
first_run_img = load_img(str(first_run_path))
ref_img = index_img(first_run_img, 0)

s1_indices = [
    i for i, lab in enumerate(atlas.labels)
    if "L G_postcentral" in str(lab) and i != 0
]

atlas_img = load_img(atlas.maps)
atlas_data = atlas_img.get_fdata()
mask_data = np.isin(atlas_data, s1_indices).astype("uint8")
s1_mask = new_img_like(atlas_img, mask_data)
s1_mask_resampled = resample_to_img(s1_mask, ref_img, interpolation="nearest")

masker = NiftiMasker(mask_img=s1_mask_resampled, standardize=False)
masker.fit(first_run_img)

N_FEATURES = int(np.sum(s1_mask_resampled.get_fdata() > 0))
print(f"S1 voxels: {N_FEATURES}")
print("Selected atlas regions:")
for i in s1_indices:
    print(f"  - {atlas.labels[i]}")

[fetch_atlas_destrieux_2009] Dataset found in /home/duarte/nilearn_data/destrieux_2009


/tmp/ipykernel_52216/1867691523.py:5: UserWarning: 
The following regions are present in the atlas look-up table,
but missing from the atlas image:

 index          name
    42 L Medial_wall
   117 R Medial_wall

  atlas = fetch_atlas_destrieux_2009()


S1 voxels: 2187
Selected atlas regions:
  - L G_postcentral


/tmp/ipykernel_52216/1867691523.py:26: UserWarning: [NiftiMasker.fit] Generation of a mask has been requested (imgs != None) while a mask was given at masker creation. Given mask will be used.
  masker.fit(first_run_img)


In [15]:
import torch
import torch.nn as nn

class SomatotopicANN(nn.Module):
    def __init__(self, input_size: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 4),
        )

    def forward(self, x):
        return self.net(x)

In [16]:
LEARNING_RATE  = 0.001
WEIGHT_DECAY   = 0.001
DROPOUT        = 0.3
BATCH_SIZE     = 32
MAX_EPOCHS     = 500
PATIENCE       = 100  

print(f"ANN {N_FEATURES}→64→16→4 | lr={LEARNING_RATE} | wd={WEIGHT_DECAY} | "
      f"dropout={DROPOUT} | batch={BATCH_SIZE} | max_epochs={MAX_EPOCHS}")

ANN 2187→64→16→4 | lr=0.001 | wd=0.001 | dropout=0.3 | batch=32 | max_epochs=500


In [ ]:
import copy, time, pickle
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                              confusion_matrix, precision_recall_fscore_support)
import matplotlib.pyplot as plt
import seaborn as sns

all_subjects_results = []

for subject in subjects:
    print(f"\n{'='*60}\n{subject}")
    t0 = time.time()

    subj_dir    = RESULTS_BASE_DIR / subject
    figures_dir = subj_dir / "figures"
    models_dir  = subj_dir / "models"
    figures_dir.mkdir(parents=True, exist_ok=True)
    models_dir.mkdir(parents=True, exist_ok=True)

    #1. Load events
    all_events = []
    for run in range(1, n_runs_per_subject + 1):
        ev_path = BIDS_ROOT / "events" / f"{subject}_{session}_{task}_run-{run}_events.tsv"
        df = pd.read_csv(ev_path, sep="\t")
        df["run"] = run
        all_events.append(df)
    events_df   = pd.concat(all_events, ignore_index=True)
    stim_events = events_df[~events_df["trial_type"].isin(["Baseline", "Jitter"])].copy()
    stim_events["region"] = stim_events["trial_type"].map(ELECTRODE_TO_REGION)
    print(f"  Stimulation events: {len(stim_events)}")

    #2. Extract 148-dim feature vectors
    X_list, y_list, run_list = [], [], []

    for run in range(1, n_runs_per_subject + 1):
        bold_path = (DERIVATIVES / subject / session / "func" /
                     f"{subject}_{session}_{task}_run-{run}_space-{space}_desc-preproc_bold.nii")
        bold_img  = load_img(str(bold_path))
        run_evs   = stim_events[stim_events["run"] == run]

        for _, event in run_evs.iterrows():
            vol_center  = int(np.round((event["onset"] + HRF_DELAY) / tr))
            vol_indices = [vol_center - WINDOW, vol_center, vol_center + WINDOW]

            if all(0 <= v < bold_img.shape[3] for v in vol_indices):
                # mean of 3 volumes
                parcel_vecs = [masker.transform(index_img(bold_img, v))[0]
                               for v in vol_indices]
                X_list.append(np.mean(parcel_vecs, axis=0))
                y_list.append(event["region"])
                run_list.append(run)


    X = np.vstack(X_list).astype(np.float32)
    y = np.array([REGION_TO_LABEL[r] for r in y_list])
    run_labels = np.array(run_list)

    print(f"Feature matrix: {X.shape}")
    if X.ndim != 2:
        raise ValueError(f"Expected 2D X, got shape {X.shape}")

    cw = compute_class_weight("balanced", classes=np.unique(y), y=y)
    cw_tensor = torch.FloatTensor(cw)

    #3. Leave-One-Run-Out CV
    fold_results = []
    best_bal_acc = 0.0
    best_state = None
    best_scaler = None

    for test_run in range(1, n_runs_per_subject + 1):
        train_mask = run_labels != test_run
        test_mask = run_labels == test_run

        X_tr, X_te = X[train_mask], X[test_mask]
        y_tr, y_te = y[train_mask], y[test_mask]

        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_te_s = scaler.transform(X_te)

        X_te_t = torch.FloatTensor(X_te_s)
        y_te_t = torch.LongTensor(y_te)

        loader = DataLoader(
            TensorDataset(torch.FloatTensor(X_tr_s), torch.LongTensor(y_tr)),
            batch_size=BATCH_SIZE, shuffle=True,
        )

        torch.manual_seed(RANDOM_SEED)
        model = SomatotopicANN(input_size=N_FEATURES, dropout=DROPOUT)
        optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                                weight_decay=WEIGHT_DECAY)
        criterion = nn.CrossEntropyLoss(weight=cw_tensor)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, 
            mode="max", 
            factor=0.5, 
            patience=PATIENCE, 
            min_lr=LEARNING_RATE
        )

        best_fold_bal = 0.0
        best_fold_state = None
        no_improve = 0

        for epoch in range(MAX_EPOCHS):
            model.train()
            for bX, by in loader:
                optimizer.zero_grad()
                criterion(model(bX), by).backward()
                optimizer.step()

            model.eval()
            with torch.no_grad():
                preds = model(X_te_t).argmax(dim=1).numpy()
            bal = balanced_accuracy_score(y_te, preds) * 100
            scheduler.step(bal)

            if bal > best_fold_bal:
                best_fold_bal   = bal
                best_fold_state = copy.deepcopy(model.state_dict())
                no_improve      = 0
            else:
                no_improve += 1
            if no_improve >= PATIENCE:
                break

        model.load_state_dict(best_fold_state)
        model.eval()
        with torch.no_grad():
            y_pred = model(X_te_t).argmax(dim=1).numpy()

        acc = accuracy_score(y_te, y_pred)
        bal = balanced_accuracy_score(y_te, y_pred)
        cm  = confusion_matrix(y_te, y_pred, labels=list(range(4)))
        prec, rec, f1, _ = precision_recall_fscore_support(
            y_te, y_pred, labels=list(range(4)), average=None, zero_division=0
        )
        fold_results.append({"fold": test_run, "acc": acc, "bal": bal,
                              "cm": cm, "prec": prec, "rec": rec, "f1": f1})
        print(f"  Fold {test_run}: acc={acc*100:.1f}%  bal={bal*100:.1f}%  ({epoch+1} ep)")

        if bal > best_bal_acc:
            best_bal_acc = bal
            best_state   = copy.deepcopy(best_fold_state)
            best_scaler  = copy.deepcopy(scaler)

    #4. Aggregate folds
    cm_total = sum(r["cm"] for r in fold_results)
    mean_acc = np.mean([r["acc"] for r in fold_results])
    std_acc  = np.std([r["acc"] for r in fold_results])
    mean_bal = np.mean([r["bal"] for r in fold_results])
    std_bal  = np.std([r["bal"] for r in fold_results])
    mean_prec = np.mean([r["prec"] for r in fold_results], axis=0)
    mean_rec = np.mean([r["rec"] for r in fold_results], axis=0)
    mean_f1 = np.mean([r["f1"] for r in fold_results], axis=0)
    print(f"  → acc={mean_acc*100:.1f}±{std_acc*100:.1f}%  "
          f"bal={mean_bal*100:.1f}±{std_bal*100:.1f}%  "
          f"time={time.time()-t0:.0f}s")

    #5. Save model, scaler, confusion matrix
    torch.save(best_state, models_dir / "best_model.pt")
    np.save(models_dir / "confusion_matrix.npy", cm_total)
    with open(models_dir / "scaler.pkl", "wb") as f:
        pickle.dump(best_scaler, f)

    #6. Subject-level confusion matrix figure
    class_names = [LABEL_TO_REGION[i] for i in range(4)]
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm_total, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f"{subject}\nbal.acc = {mean_bal*100:.1f}%")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.tight_layout()
    plt.savefig(figures_dir / "confusion_matrix.png", dpi=150)
    plt.close()

    all_subjects_results.append({
        "subject": subject,
        "mean_accuracy": mean_acc,
        "std_accuracy": std_acc,
        "mean_balanced_accuracy": mean_bal,
        "std_balanced_accuracy": std_bal,
        "macro_precision": float(np.mean(mean_prec)),
        "macro_recall": float(np.mean(mean_rec)),
        "macro_f1": float(np.mean(mean_f1)),
        "per_class_precision": str(mean_prec.tolist()),
        "per_class_recall": str(mean_rec.tolist()),
        "per_class_f1": str(mean_f1.tolist()),
    })

print(f"\nAll {len(subjects)} subjects done.")


sub-p0002
  Stimulation events: 160
  Feature matrix: (160,)


ValueError: Expected 2D array, got 1D array instead:
array=[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [ ]:
results_df = pd.DataFrame(all_subjects_results)
results_df.to_csv(RESULTS_BASE_DIR / "all_subjects_results.csv", index=False)

print(
    results_df[["subject", "mean_balanced_accuracy",
                "std_balanced_accuracy", "macro_f1"]].to_string(index=False)
)

fig, ax = plt.subplots(figsize=(max(6, len(results_df) * 1.2), 4))
x = range(len(results_df))
ax.bar(
    x,
    results_df["mean_balanced_accuracy"] * 100,
    yerr=results_df["std_balanced_accuracy"] * 100,
    capsize=5,
    color="steelblue",
    ecolor="black",
)
ax.axhline(25, color="red", linestyle="--", label="Chance (25%)")
ax.set_xticks(list(x))
ax.set_xticklabels(results_df["subject"], rotation=45, ha="right")
ax.set_ylabel("Balanced Accuracy (%)")
ax.set_title(f"ANN ({N_FEATURES}→64→16→4) | S1 ROI | LORO-CV")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_BASE_DIR / "group_balanced_accuracy.png", dpi=150)
plt.show()

print(f"\nCSV  → {(RESULTS_BASE_DIR / 'all_subjects_results.csv').resolve()}")
print(f"Plot → {(RESULTS_BASE_DIR / 'group_balanced_accuracy.png').resolve()}")